In [ ]:
from segmentation_models import Unet, Linknet, FPN, PSPNet
from segmentation_models import get_preprocessing, backbones
from tensorflow import layers

def create_model():
    BACKBONE = 'efficientnetb7'  # or your specific variant like efficientnetb0
    model1 = Unet(BACKBONE, input_shape=(128, 128, 3), encoder_weights='imagenet')  
    c = optimizers.adam(lr = 0.001)
    model1.compile(loss=bce_dice_loss, optimizer=c, metrics=[my_iou_metric])
    return model1

In [ ]:
client_data_train = ((x_train1, y_train1), (x_train2, y_train2), (x_train3, y_train3))
client_data_test = ((x_valid1, y_valid1), (x_valid2, y_valid2), (x_valid3, y_valid3))

## FedAvg

In [ ]:
global_model = create_model()

local_model = create_model()

num_rounds = 250
num_clients = 3

# History tracking
global_iou_history = []
client_iou_history = [[] for _ in range(num_clients)]
temp_global_model_weights = global_model.get_weights()
# Calculate sample counts for weighted averaging
train_counts = [len(client_data_train[i][0]) for i in range(num_clients)]
test_counts = [len(client_data_test[i][0]) for i in range(num_clients)]
total_train_samples = sum(train_counts)
total_test_samples = sum(test_counts)

best_global_iou = 0.0

for round_num in range(num_rounds):
    local_weights = []
    local_ious = []

    # Client-side Training and Local Validation
    for client_id in range(num_clients):
        # Sync local model with current global weights
        local_model.set_weights(global_model.get_weights())

        # Get local training and testing data
        x_train_local, y_train_local = client_data_train[client_id]
        x_test_local, y_test_local = client_data_test[client_id]

        # Local Training
        with tf.device('/GPU:0'):
            local_model.fit(x_train_local, y_train_local, epochs=1, batch_size=32, verbose=0)
        
        # Local Evaluation (No shared data access)
        _, iou, accuracy = local_model.evaluate(x_test_local, y_test_local, verbose=0)
        
        # Record local results
        local_ious.append(iou)
        client_iou_history[client_id].append(iou)
        local_weights.append(local_model.get_weights())

    # Server-side Aggregation
    # 1. Federated Averaging of Weights (based on training samples)
    new_global_weights = [
        np.sum([local_weights[c][layer] * (train_counts[c] / total_train_samples) 
                for c in range(num_clients)], axis=0)
        for layer in range(len(local_weights[0]))
    ]
    global_model.set_weights(new_global_weights)

    # 2. Federated Aggregation of IoU (based on testing samples)
    aggregated_global_iou = sum([local_ious[c] * (test_counts[c] / total_test_samples) 
                                 for c in range(num_clients)])
    global_iou_history.append(aggregated_global_iou)

    # Checkpoint the best global model
    if aggregated_global_iou > best_global_iou:
        best_global_iou = aggregated_global_iou
        # Save model or weights here
        temp_global_model_weights =  global_model.get_weights()
        print(f"Round {round_num + 1}: Global IoU improved to {aggregated_global_iou:.4f}")
    else:
        print(f"Round {round_num + 1}: Global IoU {aggregated_global_iou:.4f}")

## FedProx

In [ ]:
# FedProx Hyperparameter (mu). 
# mu=0 recovers FedAvg. Higher mu increases regularization.
mu = 0.001 

num_rounds = 100
num_clients = 3

global_iou_history = []
client_iou_history = [[] for _ in range(num_clients)]

train_counts = [len(client_data_train[i][0]) for i in range(num_clients)]
test_counts = [len(client_data_test[i][0]) for i in range(num_clients)]
total_train_samples = sum(train_counts)
total_test_samples = sum(test_counts)

best_global_iou = 0.0

# Define optimizer and loss outside the loop
optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.BinaryCrossentropy()

for round_num in range(num_rounds):
    local_weights = []
    local_ious = []

    # Get current global weights to anchor the proximal term
    current_global_weights = global_model.get_weights()
    # Convert to tensors for faster computation during the custom loop
    global_vars = [tf.convert_to_tensor(w) for w in current_global_weights]

    for client_id in range(num_clients):
        local_model.set_weights(current_global_weights)
        x_train_local, y_train_local = client_data_train[client_id]
        x_test_local, y_test_local = client_data_test[client_id]

        # Custom training loop for FedProx
        # You can wrap this in a @tf.function for speed
        with tf.device('/GPU:0'):
            # Convert local data to tensor slices for the custom loop
            train_ds = tf.data.Dataset.from_tensor_slices((x_train_local, y_train_local)).batch(32)
            
            for x_batch, y_batch in train_ds:
                with tf.GradientTape() as tape:
                    predictions = local_model(x_batch, training=True)
                    original_loss = loss_fn(y_batch, predictions)
                    
                    # Calculate Proximal Term: (mu / 2) * ||w - w_t||^2
                    proximal_term = 0.0
                    for local_var, global_var in zip(local_model.trainable_variables, global_vars):
                        proximal_term += tf.nn.l2_loss(local_var - global_var)
                    
                    total_loss = original_loss + (mu * proximal_term)
                
                gradients = tape.gradient(total_loss, local_model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, local_model.trainable_variables))
        
        # Evaluate local model performance
        _, iou, accuracy = local_model.evaluate(x_test_local, y_test_local, verbose=0)
        local_ious.append(iou)
        client_iou_history[client_id].append(iou)
        local_weights.append(local_model.get_weights())

    # Server-side aggregation (FedAvg remains the same for weight aggregation)
    new_global_weights = [
        np.sum([local_weights[c][layer] * (train_counts[c] / total_train_samples) 
                for c in range(num_clients)], axis=0)
        for layer in range(len(local_weights[0]))
    ]
    global_model.set_weights(new_global_weights)

    # Calculate Aggregated Global IoU
    aggregated_global_iou = sum([local_ious[c] * (test_counts[c] / total_test_samples) 
                                 for c in range(num_clients)])
    global_iou_history.append(aggregated_global_iou)

    if aggregated_global_iou > best_global_iou:
        best_global_iou = aggregated_global_iou
        print(f"Round {round_num + 1}: Global IoU improved to {aggregated_global_iou:.4f}")
    else:
        print(f"Round {round_num + 1}: Global IoU {aggregated_global_iou:.4f}")